# 🏗️ Notebook 1: Google Search — Requirements & Architecture

## 🛠️ Setup

```bash
cd 06-system-designs/google-search
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## What we're designing

A web-scale search engine: take a text query, return the most relevant pages in <200ms.

### Functional requirements
- Crawl the web, follow links, fetch pages.
- **Index** pages by the words they contain.
- Given a query, return the top-K most relevant documents.
- Rank by relevance + quality + personalization.
- Suggestions (typeahead) — see the typeahead lab.

### Non-functional
- **Huge scale**: 50B+ pages, trillions of terms, hundreds of millions of queries/day.
- **Low latency** for queries (<200ms).
- Reasonable **freshness** (news within minutes, long-tail within weeks).


## Three pipelines

```
   ┌──────────┐   fetch  ┌──────────┐  parse  ┌──────────┐
   │ Crawler  │─────────▶│ Document │────────▶│ Indexer  │
   └──────────┘          │  Store   │         └──────────┘
        ▲                └──────────┘              │
        │ seed + discovered                        ▼
        │                                   ┌────────────┐
        │                                   │ Inverted   │
        │                                   │ Index      │
        │                                   └────────────┘
        │                                         ▲
        │                                         │ lookup
        │                                   ┌────────────┐
        └─────── Ranking/Quality signals ───│ Query Svc  │◀── user
                                            └────────────┘
```

Three *independent* pipelines:
1. **Crawling** — build a document corpus.
2. **Indexing** — preprocess docs into an inverted index.
3. **Serving** — answer queries against the index.


## Back-of-envelope

- 50B pages × 100 KB = **5 PB** of raw HTML.
- Inverted index (compressed): ~10–20% of raw → ~500 TB.
- Sharded across 1000s of machines; each shard holds a slice of the vocabulary or doc-id range.
- Query: hits every shard in parallel, partial results merged.
